# Day 5-3: 모델 최적화 & 배포

**강의 시간**: 1.5시간  
**학습 목표**:
- TFLite 변환 (INT8 Quantization)
- 모델 크기 비교 (Keras vs TFLite)
- 정적 이미지 테스트
- 로컬 실시간 데모 스크립트 제공

**사전 요구사항**: Day 5-2 완료 (MLflow에 모델 저장)  
**최종 목표**: 모바일 배포 가능한 경량 모델

## 🔧 0. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow opencv-python scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import time
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras

import mlflow
import dagshub

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU: {len(tf.config.list_physical_devices('GPU'))} devices")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# MLflow 설정
import mlflow
import dagshub

# 🔥 본인의 정보로 수정!
repo_owner = # 🔥 직접 작성이 필요합니다.  # 본인의 Dagshub username
repo_name  = # 🔥 직접 작성이 필요합니다.  # 생성한 repository 이름 (예: deeplearning-bootcamp)

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day5-gesture-recognition')

print("✅ MLflow 설정 완료!")

## 📦 1. MLflow에서 Best 모델 로드

Day 5-2에서 학습한 MobileNetV2 모델을 MLflow에서 불러옵니다.

In [ ]:
# MLflow 실험 조회
experiment = mlflow.get_experiment_by_name('day5-gesture-recognition')

if experiment is None:
    print("❌ 실험을 찾을 수 없습니다!")
    print("   Day 5-2를 먼저 실행하세요.")
else:
    print(f"✅ 실험 발견: {experiment.name}")
    print(f"   Experiment ID: {experiment.experiment_id}")

In [ ]:
# Phase 2 모델 찾기
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'MobileNetV2_Phase2_FineTuning'",
    order_by=["start_time DESC"],
    max_results=1
)

if len(runs) == 0:
    print("❌ Phase 2 모델을 찾을 수 없습니다!")
    print("   Day 5-2를 먼저 실행하세요.")
else:
    run = runs.iloc[0]
    run_id = run['run_id']

    print("="*60)
    print("  Best Model 발견")
    print("="*60)
    print(f"Run Name: {run['tags.mlflow.runName']}")
    print(f"Run ID: {run_id}")
    print(f"Val Accuracy: {run['metrics.phase2_final_val_accuracy']:.4f} ({run['metrics.phase2_final_val_accuracy']*100:.2f}%)")
    print(f"Model Size: {run['metrics.model_size_mb']:.2f} MB")
    print("="*60)

In [ ]:
# 모델 로드
model_uri = f"runs:/{run_id}/model"
model = mlflow.keras.load_model(model_uri)

print("\n✅ 모델 로드 완료!")
print(f"   Parameters: {model.count_params():,}")
print(f"   Input Shape: {model.input_shape}")
print(f"   Output Shape: {model.output_shape}")

In [ ]:
# 제스처 이름 정의 (19클래스: no_gesture 포함 시 19, 아닐 경우 18)
gesture_names = [
    'call', 'dislike', 'fist', 'four', 'like', 'mute',
    'ok', 'one', 'palm', 'peace', 'peace_inverted', 'rock',
    'stop', 'stop_inverted', 'three', 'three2', 'two_up', 'two_up_inverted'
]

print(f"\n제스처 클래스: {len(gesture_names)}개")
print(f"{gesture_names}")

## 🔄 2. TensorFlow Lite 변환 (INT8 Quantization)

In [ ]:
# Keras 모델 저장 (TFLite 변환용)
model.save('mobilenetv2_gesture_fp32.h5')

keras_size = os.path.getsize('mobilenetv2_gesture_fp32.h5') / (1024**2)
print(f"✅ Keras 모델 저장 완료!")
print(f"   Size: {keras_size:.2f} MB")

In [ ]:
# Representative Dataset 준비 (Quantization용)
data_dir = Path('./data/hagrid-classification-512p-no-gesture-150k')

if not data_dir.exists():
    print("⚠️ 데이터 폴더가 없습니다!")
    print("   Quantization을 위해서는 Day 5-1 데이터가 필요합니다.")
    print("   지금은 Dynamic Range Quantization만 수행합니다.")
    use_quantization = False
else:
    print("✅ 데이터 폴더 확인")
    use_quantization = True

In [ ]:
# TFLite Converter 설정
converter = tf.lite.TFLiteConverter.from_keras_model(model)

if use_quantization:
    print("🔧 INT8 Quantization 설정 중...")

    # Optimization 설정
    converter.optimizations = # 🔥 직접 작성이 필요합니다.  # [tf.lite.Optimize.DEFAULT]

    # Representative dataset 생성
    def representative_dataset():
        """Quantization을 위한 대표 데이터셋"""
        gesture_folders = sorted([d for d in data_dir.iterdir() if d.is_dir()])

        count = 0
        for gesture_folder in gesture_folders:
            image_paths = list(gesture_folder.glob('*.jpeg'))[:10]  # 각 클래스당 10개

            for img_path in image_paths:
                # 이미지 로드
                img = cv2.imread(str(img_path))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (224, 224))

                # MobileNetV2 전처리
                img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
                img = np.expand_dims(img, axis=0).astype(np.float32)

                # 🔥 직접 작성이 필요합니다.  # yield [img]

                count += 1
                if count >= 100:  # 총 100개 샘플
                    return

    converter.representative_dataset = # 🔥 직접 작성이 필요합니다.  # representative_dataset

    # INT8 설정
    converter.target_spec.supported_ops = # 🔥 직접 작성이 필요합니다.  # [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type  = # 🔥 직접 작성이 필요합니다.  # tf.uint8
    converter.inference_output_type = # 🔥 직접 작성이 필요합니다.  # tf.uint8

    print("✅ INT8 Quantization 설정 완료!")
else:
    print("🔧 Dynamic Range Quantization만 수행...")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

print("\n변환 중... (1-2분 소요)")

In [ ]:
# TFLite 변환 실행
try:
    tflite_model = converter.convert()

    # 저장
    tflite_filename = 'mobilenetv2_gesture_int8.tflite' if use_quantization else 'mobilenetv2_gesture_dynamic.tflite'
    with open(tflite_filename, 'wb') as f:
        f.write(tflite_model)

    tflite_size = len(tflite_model) / (1024**2)

    print("\n✅ TFLite 변환 완료!")
    print(f"   파일: {tflite_filename}")
    print(f"   Size: {tflite_size:.2f} MB")
    print(f"   압축률: {keras_size / tflite_size:.1f}x")

except Exception as e:
    print(f"\n❌ 변환 실패: {e}")

## 📊 3. 모델 크기 비교

In [ ]:
# 크기 비교 시각화
models = ['Keras\n(FP32)', 'TFLite\n(Quantized)']
sizes = [keras_size, tflite_size]
colors = ['#ff9999', '#99ccff']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, sizes, color=colors, edgecolor='black', linewidth=2)

# 값 표시
for i, (bar, size) in enumerate(zip(bars, sizes)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{size:.2f} MB',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_ylabel('모델 크기 (MB)', fontsize=12, fontweight='bold')
ax.set_title(f'모델 크기 비교 (압축률: {keras_size/tflite_size:.1f}x)',
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_size_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n압축 효과:")
print(f"  원본: {keras_size:.2f} MB")
print(f"  압축: {tflite_size:.2f} MB")
print(f"  절감: {keras_size - tflite_size:.2f} MB ({(1 - tflite_size/keras_size)*100:.1f}%)")

## 🧪 4. TFLite 모델 추론 테스트

In [ ]:
# TFLite Interpreter 로드
interpreter = # 🔥 직접 작성이 필요합니다.  # tf.lite.Interpreter(model_path=tflite_filename)
# 🔥 직접 작성이 필요합니다.  # interpreter.allocate_tensors()

# 입력/출력 정보
input_details  = # 🔥 직접 작성이 필요합니다.  # interpreter.get_input_details()
output_details = # 🔥 직접 작성이 필요합니다.  # interpreter.get_output_details()

print("="*60)
print("  TFLite Interpreter 정보")
print("="*60)
print(f"Input Shape: {input_details[0]['shape']}")
print(f"Input Type: {input_details[0]['dtype']}")
print(f"Output Shape: {output_details[0]['shape']}")
print(f"Output Type: {output_details[0]['dtype']}")
print("="*60)

In [ ]:
# 테스트 이미지 준비
if data_dir.exists():
    # 각 제스처별로 1개씩 샘플
    test_images = []
    test_labels = []

    gesture_folders = sorted([d for d in data_dir.iterdir() if d.is_dir()])

    for i, gesture_folder in enumerate(gesture_folders):
        image_paths = list(gesture_folder.glob('*.jpeg'))
        if len(image_paths) > 0:
            img_path = image_paths[0]
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))

            test_images.append(img)
            test_labels.append(i)

    print(f"✅ 테스트 이미지 준비 완료: {len(test_images)}개")
else:
    print("⚠️ 테스트 이미지가 없습니다.")
    test_images = []
    test_labels = []

In [ ]:
# Keras vs TFLite 비교
if len(test_images) > 0:
    keras_preds = []
    tflite_preds = []

    for img in test_images:
        # Keras 추론
        img_keras = tf.keras.applications.mobilenet_v2.preprocess_input(img.copy())
        img_keras = np.expand_dims(img_keras, axis=0)
        pred_keras = model.predict(img_keras, verbose=0)
        keras_preds.append(np.argmax(pred_keras))

        # TFLite 추론
        if use_quantization:
            img_tflite = img.astype(np.uint8)
        else:
            img_tflite = tf.keras.applications.mobilenet_v2.preprocess_input(img.copy())
            img_tflite = img_tflite.astype(np.float32)

        img_tflite = np.expand_dims(img_tflite, axis=0)

        interpreter.set_tensor(input_details[0]['index'], img_tflite)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])

        if use_quantization:
            scale, zero_point = output_details[0]['quantization']
            output_data = scale * (output_data.astype(np.float32) - zero_point)

        tflite_preds.append(np.argmax(output_data))

    # 정확도 비교
    keras_acc  = np.mean(np.array(keras_preds) == np.array(test_labels))
    tflite_acc = np.mean(np.array(tflite_preds) == np.array(test_labels))

    print("\n" + "="*60)
    print("  Keras vs TFLite 정확도 비교")
    print("="*60)
    print(f"Keras (FP32): {keras_acc*100:.2f}%")
    print(f"TFLite ({'INT8' if use_quantization else 'Dynamic'}): {tflite_acc*100:.2f}%")
    print(f"차이: {abs(keras_acc - tflite_acc)*100:.2f}%p")
    print("="*60)

    if abs(keras_acc - tflite_acc) < 0.02:
        print("\n✅ 정확도 손실 < 2%: 성공적인 변환!")
    else:
        print("\n⚠️ 정확도 손실 > 2%: 추가 튜닝 필요")
else:
    print("테스트를 건너뜁니다.")

## ⚡ 5. 추론 속도 비교

In [ ]:
# Keras 추론 속도
if len(test_images) > 0:
    sample_img = test_images[0]
    sample_img_keras = tf.keras.applications.mobilenet_v2.preprocess_input(sample_img.copy())
    sample_img_keras = np.expand_dims(sample_img_keras, axis=0)

    # Warmup
    for _ in range(10):
        _ = model.predict(sample_img_keras, verbose=0)

    # 측정
    keras_times = []
    for _ in range(100):
        start = time.time()
        _ = model.predict(sample_img_keras, verbose=0)
        keras_times.append((time.time() - start) * 1000)

    keras_latency = np.mean(keras_times)
    keras_fps = 1000 / keras_latency

    print("Keras (FP32):")
    print(f"  Latency: {keras_latency:.2f} ± {np.std(keras_times):.2f} ms")
    print(f"  FPS: {keras_fps:.1f}")

In [ ]:
# TFLite 추론 속도
if len(test_images) > 0:
    sample_img_tflite = test_images[0]
    if use_quantization:
        sample_img_tflite = sample_img_tflite.astype(np.uint8)
    else:
        sample_img_tflite = tf.keras.applications.mobilenet_v2.preprocess_input(sample_img_tflite.copy())
        sample_img_tflite = sample_img_tflite.astype(np.float32)

    sample_img_tflite = np.expand_dims(sample_img_tflite, axis=0)

    # Warmup
    for _ in range(10):
        interpreter.set_tensor(input_details[0]['index'], sample_img_tflite)
        interpreter.invoke()

    # 측정
    tflite_times = []
    for _ in range(100):
        start = time.time()
        # 🔥 직접 작성이 필요합니다.  # interpreter.set_tensor(input_details[0]['index'], sample_img_tflite)
        # 🔥 직접 작성이 필요합니다.  # interpreter.invoke()
        # 🔥 직접 작성이 필요합니다.  # _ = interpreter.get_tensor(output_details[0]['index'])
        tflite_times.append((time.time() - start) * 1000)

    tflite_latency = np.mean(tflite_times)
    tflite_fps = 1000 / tflite_latency

    print(f"\nTFLite ({'INT8' if use_quantization else 'Dynamic'}):")
    print(f"  Latency: {tflite_latency:.2f} ± {np.std(tflite_times):.2f} ms")
    print(f"  FPS: {tflite_fps:.1f}")

    print(f"\n속도 향상: {keras_latency / tflite_latency:.1f}x")

## 📊 6. 최종 결과 시각화

In [ ]:
# Day 5 전체 여정
if len(test_images) > 0:
    journey_data = {
        'Model': ['Baseline\nCNN', 'MobileNetV2\nPhase1', 'MobileNetV2\nPhase2', 'TFLite\n(Quantized)'],
        'Accuracy': [73.52, 66.47, 97.58, tflite_acc * 100],
        'Size': [8, 28, 28, tflite_size],
        'Latency': [15, 2, 1.8, tflite_latency]
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Accuracy
    axes[0].plot(journey_data['Model'], journey_data['Accuracy'],
                marker='o', linewidth=2, markersize=10, color='#2ecc71')
    axes[0].set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
    axes[0].set_title('정확도 변화', fontweight='bold', fontsize=14)
    axes[0].grid(alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    # Size
    axes[1].bar(journey_data['Model'], journey_data['Size'],
               color=['#ff9999', '#ffcc99', '#ffcc99', '#99ccff'], edgecolor='black')
    axes[1].set_ylabel('Model Size (MB)', fontweight='bold', fontsize=12)
    axes[1].set_title('모델 크기 변화', fontweight='bold', fontsize=14)
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)

    # Latency
    axes[2].plot(journey_data['Model'], journey_data['Latency'],
                marker='o', linewidth=2, markersize=10, color='#e74c3c')
    axes[2].set_ylabel('Latency (ms)', fontweight='bold', fontsize=12)
    axes[2].set_title('추론 속도 변화', fontweight='bold', fontsize=14)
    axes[2].grid(alpha=0.3)
    axes[2].tick_params(axis='x', rotation=45)

    plt.suptitle('Day 5 전체 여정: Baseline → Quantized TFLite',
                fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('day5_journey.png', dpi=100, bbox_inches='tight')
    plt.show()

## 💻 7. 로컬 실시간 데모 스크립트 생성

Colab에서는 웹캠 접근이 제한적이므로, 로컬 PC에서 실행할 수 있는 Python 스크립트를 생성합니다.

In [ ]:
# 로컬 실행용 스크립트 생성
demo_script = '#!/usr/bin/env python3\n"""\nHaGRID 손 제스처 인식 - 실시간 데모\n실행 방법: python local_demo.py\n종료: \'q\' 키\n"""\n\nimport cv2\nimport numpy as np\nimport tensorflow as tf\nimport time\n\n# 모델 로드\nprint("모델 로딩 중...")\nmodel = tf.keras.models.load_model(\'mobilenetv2_gesture_fp32.h5\')\nprint("✅ 모델 로드 완료!")\n\ngesture_names = [\n    \'call\', \'dislike\', \'fist\', \'four\', \'like\', \'mute\',\n    \'ok\', \'one\', \'palm\', \'peace\', \'peace_inverted\', \'rock\',\n    \'stop\', \'stop_inverted\', \'three\', \'three2\', \'two_up\', \'two_up_inverted\'\n]\n\ncap = cv2.VideoCapture(0)\ncap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)\ncap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)\n\nif not cap.isOpened():\n    print("❌ 웹캠을 열 수 없습니다!")\n    exit()\n\nprint("✅ 웹캠 시작!")\nprint("   \'q\' 키를 눌러 종료")\n\nfps_history = []\nframe_count = 0\nstart_time = time.time()\n\nwhile True:\n    ret, frame = cap.read()\n    if not ret:\n        break\n\n    img = cv2.resize(frame, (224, 224))\n    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n    img_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(img_rgb)\n    img_batch = np.expand_dims(img_preprocessed, axis=0)\n\n    inference_start = time.time()\n    pred = model.predict(img_batch, verbose=0)\n    inference_time = (time.time() - inference_start) * 1000\n\n    gesture_idx = np.argmax(pred)\n    gesture = gesture_names[gesture_idx] if gesture_idx < len(gesture_names) else f"Unknown ({gesture_idx})"\n    confidence = pred[0][gesture_idx]\n\n    frame_count += 1\n    if frame_count % 30 == 0:\n        elapsed = time.time() - start_time\n        fps_history.append(frame_count / elapsed)\n\n    current_fps = fps_history[-1] if fps_history else 0\n\n    cv2.putText(frame, f\'Gesture: {gesture}\',\n                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)\n    cv2.putText(frame, f\'Confidence: {confidence:.2f}\',\n                (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)\n    cv2.putText(frame, f\'Latency: {inference_time:.1f}ms\',\n                (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)\n    cv2.putText(frame, f\'FPS: {current_fps:.1f}\',\n                (10, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)\n    cv2.rectangle(frame, (10, 200), (234, 424), (0, 255, 0), 2)\n\n    cv2.imshow(\'HaGRID Gesture Recognition\', frame)\n\n    if cv2.waitKey(1) & 0xFF == ord(\'q\'):\n        break\n\ncap.release()\ncv2.destroyAllWindows()\nprint("\\n✅ 데모 종료")\nif fps_history:\n    print(f"평균 FPS: {np.mean(fps_history):.1f}")\n'

# 스크립트 저장
with open('local_demo.py', 'w') as f:
    f.write(demo_script)

print("✅ 로컬 실행용 스크립트 생성 완료!")
print("   파일: local_demo.py")

In [ ]:
# 필요한 파일 목록
print("\n" + "="*60)
print("  로컬 실행에 필요한 파일")
print("="*60)
print("1. local_demo.py          - 실행 스크립트")
print("2. mobilenetv2_gesture_fp32.h5  - Keras 모델")
print("="*60)
print("\n실행 방법:")
print("  1. 위 파일들을 로컬 PC에 다운로드")
print("  2. Python 환경 준비:")
print("     pip install tensorflow opencv-python")
print("  3. 실행:")
print("     python local_demo.py")
print("  4. 'q' 키로 종료")
print("="*60)

In [ ]:
# 파일 다운로드 제공
from google.colab import files

print("\n📥 파일 다운로드 시작...")

try:
    files.download('local_demo.py')
    print("✅ local_demo.py 다운로드 완료!")
except:
    print("⚠️ local_demo.py 다운로드 실패")

try:
    files.download('mobilenetv2_gesture_fp32.h5')
    print("✅ mobilenetv2_gesture_fp32.h5 다운로드 완료!")
except:
    print("⚠️ mobilenetv2_gesture_fp32.h5 다운로드 실패")

try:
    files.download(tflite_filename)
    print(f"✅ {tflite_filename} 다운로드 완료!")
except:
    print(f"⚠️ {tflite_filename} 다운로드 실패")

print("\n✅ 다운로드 완료!")

## 🎉 8. Day 5 완료!

### 전체 달성 사항

**Day 5-1: EDA & Baseline**
- ✅ 153K 이미지 로드
- ✅ 18개 제스처 분류
- ✅ Baseline: 73.5%

**Day 5-2: MobileNetV2 Transfer Learning**
- ✅ 2-Phase Training
- ✅ Data Augmentation
- ✅ Val Accuracy: 97.6%

**Day 5-3: 최적화 & 배포**
- ✅ TFLite 변환
- ✅ INT8 Quantization
- ✅ 모델 크기: 28MB → 7MB
- ✅ 로컬 데모 스크립트 생성

## ✅ Day 5-3 완료 체크리스트

- [ ] MLflow에서 Best 모델 로드
- [ ] TFLite 변환 (INT8 Quantization)
- [ ] 모델 크기 비교 (~28MB → ~7MB)
- [ ] TFLite 추론 테스트
- [ ] Keras vs TFLite 정확도 비교 (< 2% 손실)
- [ ] Keras vs TFLite 추론 속도 비교
- [ ] Day 5 전체 여정 시각화
- [ ] 로컬 실행용 스크립트 생성
- [ ] 필요한 파일 다운로드
- [ ] Day 5 완료!